Objetivo:
Criar uma tabela de produtos já unindo categoria, marca(brands) 
e quantidade total de estoque em todas as lojas

In [0]:
bronze_path   = '/Volumes/bikestore/resource/bronze/'
silver_path   = '/Volumes/bikestore/resource/silver/'
gold_path     = '/Volumes/bikestore/resource/gold/'
resource_path = '/Workspace/Users/wesllan2000@yahoo.com.br/project_databricks_bikestore/resource/origem'


In [0]:
#criar dicionario path
bronze_map={
    'tmp_bronze_brands' : f'{bronze_path}/brands/',
    'tmp_bronze_categories' : f'{bronze_path}/categories/',
    #'tmp_bronze_customers' : f'{bronze_path}/customers/',
    #'tmp_bronze_order_items' : f'{bronze_path}/order_items/',
    #'tmp_bronze_orders' : f'{bronze_path}/orders/',
    'tmp_bronze_products' : f'{bronze_path}/products/',
    #'tmp_bronze_staffs' : f'{bronze_path}/staffs/',
    'tmp_bronze_stocks' : f'{bronze_path}/stocks/',
    #'tmp_bronze_stores' : f'{bronze_path}/stores/'
}
for view_name, path in bronze_map.items():
    (
        spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )

In [0]:
df_product_silver = spark.sql(
    """
with stock as (
  select
    product_id,
    sum(quantity) as total_stock
  from
    tmp_bronze_stocks
  group by
    product_id
)
select
  p.product_id,
  p.product_name,
  b.brand_name,
  c.category_name,
  p.model_year,
  p.list_price,
  s.total_stock
from
  tmp_bronze_products as p
    left join tmp_bronze_categories as c
      on p.category_id = c.category_id
    left join tmp_bronze_brands as b
      on p.brand_id = b.brand_id
    left join stock as s
      on P.product_id = s.product_id
"""
)

In [0]:
df_product_silver.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema','true')\
    .save(f'{silver_path}/product')

In [0]:
%sql
--criar tabela na silver em bikestore.logistics
CREATE OR REPLACE TABLE bikestore.logistics.silver_product 
AS (
with stock as (
  select
    product_id,
    sum(quantity) as total_stock
  from
    tmp_bronze_stocks
  group by
    product_id
)
select
  p.product_id,
  p.product_name,
  b.brand_name,
  c.category_name,
  p.model_year,
  p.list_price,
  s.total_stock
from
  tmp_bronze_products as p
    left join tmp_bronze_categories as c
      on p.category_id = c.category_id
    left join tmp_bronze_brands as b
      on p.brand_id = b.brand_id
    left join stock as s
      on P.product_id = s.product_id
)

